<a href="https://colab.research.google.com/github/angelms2003/FernandezMartinezPolo-EML-RL/blob/main/Entornos_Complejos/Q_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q-Learning

*Description*: En este notebook se desarrolla la implementación del método de **Q-Learning**, y se emplea sobre el entorno Taxi-v3 de Gymnasium.


    Autores: David Fernández Expósito
             Ángel Martínez Sánchez
             Javier Polo Gambín

    Emails: dfernandezexposito@um.es
            angel.martinezs@um.es
            javier.polog@um.es
            
    Date: 2026/02/25

Empezamos instalando e importando las librerías necesarias. También definimos los dispositivos donde se ejecutará el notebook y la semilla que vamos a usar para asegurar reproducibilidad.

In [ ]:
%%capture
!pip install gymnasium

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import gymnasium as gym
import torch
import gc
import os

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SEED = 123

# NumPy
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# Python
os.environ["PYTHONHASHSEED"] = str(SEED)

## Agente

Para implementar los distintos métodos de aprendizaje estudiados en la asignatura, hemos seguido las recomendaciones de Gymnasium para la creación de agentes y la generación de episodios, tal como se indicaba en la definición de la práctica. La idea central ha sido encapsular toda la lógica de interacción y aprendizaje en una clase `Agente`, adaptable a los diferentes algoritmos que se desean evaluar.

En este trabajo nos centramos en **Q-Learning**, un método de **Diferencias Temporales (TD)** de tipo **Off-Policy**. Al igual que SARSA, Q-Learning realiza actualizaciones paso a paso (*bootstrapping*), pero se diferencia fundamentalmente en la regla de actualización: mientras SARSA utiliza la acción que realmente va a tomar el agente ($A_{t+1}$), Q-Learning utiliza siempre la **mejor acción posible** en el siguiente estado, independientemente de lo que el agente vaya a hacer realmente.

La regla de actualización se basa directamente en la ecuación de optimalidad de Bellman:

$$
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t) \right]
$$

donde $\alpha$ es la tasa de aprendizaje y $\gamma$ es el factor de descuento. Al ser un método Off-Policy, la política de comportamiento (la que genera los datos, típicamente $\epsilon$-greedy) es diferente de la política objetivo (puramente greedy) que se está optimizando internamente.

Esta separación entre la política de comportamiento y la política objetivo confiere a Q-Learning un carácter "optimista": al respaldar siempre sus estimaciones en la acción óptima futura, el algoritmo puede ser matemáticamente más agresivo en la propagación del valor de las recompensas. Si en los primeros episodios exploratorios el agente se topa con la meta, Q-Learning asimila y retropropaga esa recompensa sin que las penalizaciones derivadas de futuras acciones exploratorias amortigüen ese conocimiento.

En cuanto a la exploración, al igual que en SARSA, el agente utiliza una política $\epsilon$-greedy con un mecanismo de decaimiento progresivo de $\epsilon$.

In [ ]:
class QLearningAgent:
    def __init__(self, env: gym.Env, alpha: float, gamma: float, epsilon: float, epsilon_decay: float, epsilon_min: float):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

        self.n_states = env.observation_space.n
        self.n_actions = env.action_space.n
        self.q_table = np.zeros((self.n_states, self.n_actions))

    def choose_action(self, state):
        """Política Epsilon-Greedy"""
        if np.random.uniform(0, 1) < self.epsilon:
            return self.env.action_space.sample()  # Exploración
        else:
            return np.argmax(self.q_table[state, :])  # Explotación

    def update(self, state, action, reward, next_state, done):
        """
        Actualización Off-Policy basada en la ecuación de Bellman.
        A diferencia de SARSA, usamos el MÁXIMO valor Q del siguiente estado,
        no el valor de la acción que realmente se va a tomar.
        """
        best_next_action = np.argmax(self.q_table[next_state, :])
        td_target = reward + self.gamma * self.q_table[next_state, best_next_action] * (not done)
        td_error = td_target - self.q_table[state, action]
        self.q_table[state, action] += self.alpha * td_error

    def decay_exploration(self):
        """Decaimiento del epsilon al final del episodio"""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def get_q_values(self):
        return self.q_table

## Esquema de aprendizaje

Ahora implementamos el proceso de aprendizaje. La función `train_agent` ejecuta el bucle de entrenamiento de Q-Learning durante un número determinado de episodios.

El flujo de Q-Learning es ligeramente más sencillo que el de SARSA: en cada paso del episodio, el agente elige una acción según su política $\epsilon$-greedy, ejecuta la transición, y actualiza $Q(S_t, A_t)$ usando directamente el máximo valor del siguiente estado. No es necesario seleccionar la siguiente acción antes de la actualización, ya que Q-Learning no depende de ella.

Esta diferencia, aunque sutil en el código, tiene implicaciones teóricas relevantes: Q-Learning puede aprender la política óptima incluso cuando la política de comportamiento es altamente exploratoria, lo que lo convierte en un método especialmente robusto.

In [ ]:
def train_agent(agent, num_episodes=5000):
    stats_rewards = []
    stats_lengths = []

    for ep in tqdm(range(num_episodes)):
        state, info = agent.env.reset(seed=SEED)
        total_reward = 0
        steps = 0
        done = False

        while not done:
            action = agent.choose_action(state)
            next_state, reward, terminated, truncated, info = agent.env.step(action)
            done = terminated or truncated

            # Q-Learning: actualización Off-Policy (usa max Q del siguiente estado)
            agent.update(state, action, reward, next_state, done)

            state = next_state
            total_reward += reward
            steps += 1

        agent.decay_exploration()
        stats_rewards.append(total_reward)
        stats_lengths.append(steps)

    return stats_rewards, stats_lengths

## Funciones auxiliares

Ahora vamos a definir una serie de funciones auxiliares que nos van a servir para mostrar resultados y realizar el análisis.

La primera función que definimos la usaremos una vez entrenado el agente, de forma que podamos evaluar el aprendizaje llevado a cabo.

In [ ]:
def capture_optimal_behavior(agente, limit_steps=100):
    """
    Graba un episodio completo siguiendo la política óptima del agente
    y devuelve las métricas de rendimiento junto con los frames de video.
    """
    visual_frames = []
    current_state, _ = agente.env.reset(seed=SEED)

    accumulated_reward = 0.0
    steps_count = 0
    is_finished = False

    while not is_finished and steps_count < limit_steps:
        img_frame = agente.env.render()
        visual_frames.append(img_frame)

        chosen_action = np.argmax(agente.get_q_values()[current_state, :])
        next_s, reward, terminated, truncated, _ = agente.env.step(chosen_action)

        accumulated_reward += reward
        current_state = next_s
        steps_count += 1
        is_finished = terminated or truncated

    visual_frames.append(agente.env.render())

    return accumulated_reward, steps_count, visual_frames

Las siguiente funciones mostrarán las gráficas de aprendizaje y longitud de los episodios una vez realizado el aprendizaje de los agentes.

La longitud del episodio es un medidor de rendimiento interesante porque no solo indica si el agente alcanza la meta, sino también **cómo de eficiente es la política aprendida**. En entornos donde existe una ruta óptima, la convergencia hacia un número estable y bajo de pasos suele reflejar que el agente ha aprendido un comportamiento estructurado y cercano al óptimo.

Además, esta métrica permite interpretar mejor los resultados: episodios muy cortos pueden indicar caídas tempranas en estados terminales negativos, mientras que episodios largos pueden reflejar exploración excesiva o movimientos erráticos. Por ello, la longitud del episodio complementa a la recompensa promedio y ayuda a entender no solo si el agente aprende, sino **cómo está aprendiendo**.

In [ ]:
def compute_running_mean(series, window):
    """Calcula un suavizado tipo media deslizante sobre una serie temporal."""
    kernel = np.full(window, 1.0 / window)
    return np.convolve(series, kernel, mode="valid")


def draw_multiple_learning_curves(results_dict, window=100, limit=None):
    """
    Representa varias curvas de entrenamiento en el mismo gráfico.
    Si se proporciona 'limit', hace zoom en los primeros N episodios.
    """
    fig, ax = plt.subplots(figsize=(10, 4))
    palette = ["darkgreen", "darkred", "navy", "orange"]

    for idx, (experiment_name, history) in enumerate(results_dict.items()):
        color = palette[idx % len(palette)]

        ax.plot(history, alpha=0.1, color=color)

        smoothed = compute_running_mean(history, window)
        ax.plot(
            np.arange(len(smoothed)),
            smoothed,
            linewidth=2,
            color=color,
            label=experiment_name
        )

    if limit:
        ax.set_xlim(0, limit)
        ax.set_title(f"Comparativa de rendimiento — Q-Learning (primeros {limit} episodios)")
    else:
        ax.set_title("Comparativa de rendimiento (Q-Learning)")

    ax.set_xlabel("Número de episodio")
    ax.set_ylabel("Recompensa por episodio")
    ax.legend()
    ax.grid(True, alpha=0.4)
    plt.show()


def draw_episode_length_comparison(length_dict, window=100, limit=None):
    """
    Compara la evolución de longitud de episodio entre varios experimentos.
    Si se proporciona 'limit', hace zoom en los primeros N episodios.
    """
    fig, ax = plt.subplots(figsize=(10, 4))
    palette = ["darkgreen", "darkred", "navy", "orange"]

    for idx, (label, values) in enumerate(length_dict.items()):
        color = palette[idx % len(palette)]

        ax.plot(values, alpha=0.1, color=color)

        smoothed = compute_running_mean(values, window)
        ax.plot(
            np.arange(len(smoothed)),
            smoothed,
            linewidth=2,
            color=color,
            label=label
        )

    if limit:
        ax.set_xlim(0, limit)
        ax.set_title(f"Longitudes de episodio — Q-Learning (primeros {limit} episodios)")
    else:
        ax.set_title("Comparativa de longitudes de episodio (Q-Learning)")

    ax.set_xlabel("Índice de episodio")
    ax.set_ylabel("Número de pasos")
    ax.legend()
    ax.grid(True, alpha=0.4)
    plt.show()

Las siguientes funciones servirán para visualizar los resultados con imágenes y gifs del comportamiento del agente.

In [ ]:
import seaborn as sns
import imageio
import base64
from IPython.display import HTML
import matplotlib.pyplot as plt

def get_taxi_qtable_directions(qtable, env):
    """
    Extrae la matriz de valores Q máximos y los símbolos de las mejores acciones
    para una configuración específica del pasajero y destino en Taxi-v3.
    """
    state, _ = env.reset(seed=SEED)
    _, _, pass_idx, dest_idx = env.unwrapped.decode(state)
    q_max_grid = np.zeros((5, 5))
    directions_grid = np.empty((5, 5), dtype=object)

    action_symbols = {0: '↓', 1: '↑', 2: '→', 3: '←', 4: 'P', 5: 'D'}

    for row in range(5):
        for col in range(5):
            state = env.unwrapped.encode(row, col, pass_idx, dest_idx)
            best_action = int(np.argmax(qtable[state]))
            max_q_value = np.max(qtable[state])
            q_max_grid[row, col] = max_q_value
            directions_grid[row, col] = action_symbols[best_action]

    return q_max_grid, directions_grid


def plot_taxi_q_values_map(qtable, env):
    '''
    Plotea un mapa de calor (Heatmap) de la política aprendida y los valores Q máximos.
    '''
    q_max_grid, directions_grid = get_taxi_qtable_directions(qtable, env)

    plt.figure(figsize=(7, 6))
    ax = sns.heatmap(
        q_max_grid,
        annot=directions_grid,
        fmt="",
        cmap=sns.color_palette("Blues", as_cmap=True),
        linewidths=1.5,
        linecolor="black",
        xticklabels=[],
        yticklabels=[],
        cbar_kws={'label': 'Max Q-Value estimado'},
        annot_kws={"fontsize": 18, "weight": "bold", "color": "black"},
    )
    ax.set_title("Learned Q-values\nArrows and letters (P/D) represent best action", fontsize=14)

    for _, spine in ax.spines.items():
        spine.set_visible(True)
        spine.set_linewidth(1.5)
        spine.set_color("black")

    plt.tight_layout()
    plt.show()


def create_gif_from_frames(frame_list, output_path="agent_taxi.gif"):
    """Genera un GIF animado a partir de una lista de imágenes."""
    with imageio.get_writer(output_path, mode="I") as gif_writer:
        for frame in frame_list:
            gif_writer.append_data(frame)
    return output_path


def show_gif_in_notebook(gif_file_path):
    """Inserta un GIF en una celda de Jupyter Notebook o Colab."""
    with open(gif_file_path, "rb") as f:
        gif_bytes = f.read()
    b64_str = base64.b64encode(gif_bytes).decode("utf-8")
    return HTML(f'<img src="data:image/gif;base64,{b64_str}" />')

## Entorno Taxi-v3

A continuación, creamos el entorno "Taxi-v3" de Gymnasium con el que trabajaremos.

`Taxi-v3` es un entorno estructurado de 500 estados utilizado para evaluar la escalabilidad de algoritmos tabulares.  

**Características del Entorno:**
* **Estados (500):** Combina 25 posiciones del taxi (cuadrícula 5x5), 5 posiciones posibles del pasajero (4 ubicaciones + dentro del taxi) y 4 destinos posibles.
* **Acciones (6):** Moverse al Sur, Norte, Este, Oeste, Recoger pasajero (Pickup) y Dejar pasajero (Dropoff).
* **Recompensas:**
  * **-1** por cada paso ejecutado (presiona al agente a encontrar la ruta más rápida).
  * **+20** por dejar al pasajero en su destino correctamente.
  * **-10** por ejecutar erróneamente *Pickup* o *Dropoff* en ubicaciones no válidas.

In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array")

Ahora creamos los diferentes agentes que usaremos. Tras analizar exhaustivamente el comportamiento de SARSA, procedemos a evaluar Q-Learning en el entorno Taxi-v3. El propósito principal de esta fase experimental consiste en determinar empíricamente si el carácter "optimista" de Q-Learning (al respaldar siempre sus estimaciones en la acción óptima futura) proporciona una ventaja tangible frente al enfoque más conservador de SARSA.

Para asegurar una comparativa transparente y aislar los efectos puramente atribuibles a la regla de actualización de cada algoritmo, se replica exactamente el mismo diseño experimental utilizado con SARSA. Se evalúan las combinaciones resultantes de $\alpha \in \{0.1, 0.5\}$ y $\gamma \in \{0.90, 0.99\}$, acompañadas del mismo decaimiento progresivo del parámetro $\epsilon$ ($\epsilon_0 = 1.0$, decay de $0.990$ por episodio, $\epsilon_{min} = 0.01$), durante $5000$ episodios.

Dada la alta velocidad de convergencia anticipada por los ensayos de SARSA, para este estudio el análisis gráfico se centrará intencionadamente en los primeros 1000 episodios, lo que permite observar con mayor detalle la dinámica inicial del aprendizaje y las posibles diferencias con SARSA en esa fase crítica.

In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array")
n_episodes = 5000

# Mismos parámetros de exploración que en SARSA para comparativa justa
epsilon_0 = 1.0
decay = 0.990
epsilon_min = 0.01

# Estudio conjunto de Q-Learning: α ∈ {0.1, 0.5} × γ ∈ {0.90, 0.99}
agents = {
    "α=0.1, γ=0.99": QLearningAgent(env, alpha=0.1, gamma=0.99, epsilon=epsilon_0, epsilon_decay=decay, epsilon_min=epsilon_min),
    "α=0.5, γ=0.99": QLearningAgent(env, alpha=0.5, gamma=0.99, epsilon=epsilon_0, epsilon_decay=decay, epsilon_min=epsilon_min),
    "α=0.1, γ=0.90": QLearningAgent(env, alpha=0.1, gamma=0.90, epsilon=epsilon_0, epsilon_decay=decay, epsilon_min=epsilon_min),
    "α=0.5, γ=0.90": QLearningAgent(env, alpha=0.5, gamma=0.90, epsilon=epsilon_0, epsilon_decay=decay, epsilon_min=epsilon_min),
}

results_rewards = {}
results_lengths = {}

for name, agent in agents.items():
    print(f"Entrenando: {name}...")
    rewards, lengths = train_agent(agent, num_episodes=n_episodes)
    results_rewards[name] = rewards
    results_lengths[name] = lengths

## Resultados

Guardamos los resultados obtenidos en diccionarios para pasárselos a las funciones auxiliares definidas anteriormente para plotear los resultados.

In [ ]:
np.savez('qlearning_results.npz', results_rewards=results_rewards, results_lengths=results_lengths)
print("Resultados guardados en 'qlearning_results.npz'")

In [ ]:
# Vista completa (5000 episodios)
draw_multiple_learning_curves(results_rewards, window=100)
draw_episode_length_comparison(results_lengths, window=100)

In [ ]:
# Zoom en los primeros 1000 episodios para observar el transitorio de aprendizaje
draw_multiple_learning_curves(results_rewards, window=100, limit=1000)
draw_episode_length_comparison(results_lengths, window=100, limit=1000)

Las gráficas de recompensa y longitud de episodio muestran que los resultados finales de Q-Learning convergen a valores prácticamente idénticos a los de SARSA. De manera análoga, la configuración que otorga el rendimiento más robusto sigue siendo $\alpha = 0.5$ junto con $\gamma = 0.99$.

Esta paridad en los resultados finales indica que, para este entorno concreto, el optimismo algorítmico de Q-Learning no aporta un salto de calidad definitivo respecto a SARSA. Analizando la estructura de Taxi-v3, esto resulta plenamente razonable: las penalizaciones drásticas (como el -10 por intentos erróneos de recoger o dejar pasajeros) son acciones fácilmente evitables en estados específicos, mientras que la navegación habitual solo incurre en una fricción constante de -1. En ausencia de zonas de penalización densas y estocásticas (como los acantilados del famoso entorno *CliffWalking*), la diferencia práctica entre aprender desde la política de comportamiento (SARSA) o desde la política óptima teórica (Q-Learning) termina difuminándose conforme $\epsilon$ decae.

Sin embargo, el zoom sobre los primeros 200 episodios revela divergencias sustanciales, particularmente vinculadas al hiperparámetro $\alpha$. Los agentes entrenados con $\alpha = 0.5$ muestran una curva de aprendizaje inicial notablemente más abrupta en Q-Learning que en SARSA, logrando reducir la longitud de los episodios con mayor celeridad. Esto se interpreta como una consecuencia directa del enfoque Off-Policy: al evaluar la ecuación de Bellman buscando siempre el máximo valor del estado siguiente, Q-Learning puede ser matemáticamente agresivo. Si en los primeros episodios exploratorios el agente se topa con la meta (obteniendo un +20) de pura casualidad, el algoritmo asimila y retropropaga esa recompensa a lo largo de la ruta sin que las penalizaciones probabilísticas derivadas de futuras acciones exploratorias amortigüen ese conocimiento.

Además, se aprecia una diferencia en la estabilidad de la convergencia tardía. La curva de aprendizaje de Q-Learning traza un perfil visiblemente más suave y menos ruidoso que la de SARSA. Esta menor varianza se explica teóricamente: mientras SARSA inyecta el ruido de sus acciones exploratorias directamente en las actualizaciones de los valores Q, Q-Learning aísla estas perturbaciones. Aunque el taxi tome puntualmente un desvío ineficiente por la política $\epsilon$-greedy durante el entrenamiento, su función de valor no se degrada, lo que ancla de manera más firme el conocimiento de la política óptima.

Ahora vamos a visualizar la política aprendida, usando las funciones auxiliares que muestran visualmente la política y generan gifs con el comportamiento del agente.

El entorno Taxi-v3 define el estado como una tupla de cuatro componentes:

$$
(row, col, passenger\_idx, destination\_idx)
$$

Esto implica que el espacio de estados es de dimensión 4, por lo que no es posible representar directamente la política completa en un mapa bidimensional.

Para poder visualizar el comportamiento aprendido por el agente, se fija la posición del pasajero y el destino, representando únicamente la política asociada a esa sección concreta del espacio de estados. De esta forma, se construye un mapa 5×5 que muestra la acción greedy seleccionada por el agente para cada posición del taxi bajo esas condiciones fijas.

Es importante destacar que esta representación no muestra la política global del agente, sino una proyección bidimensional condicionada a un estado específico del pasajero y del destino.

In [ ]:
# Mostramos el mapa de la política aprendida para el mejor agente (α=0.5, γ=0.99)
best_agent = agents["α=0.5, γ=0.99"]
plot_taxi_q_values_map(best_agent.get_q_values(), env)

Comparamos también la política del agente con peor configuración para observar las diferencias.

In [ ]:
# Política del agente con α=0.1, γ=0.90
plot_taxi_q_values_map(agents["α=0.1, γ=0.90"].get_q_values(), env)

En los mapas de calor se aprecia que el agente con la mejor configuración ($\alpha = 0.5$, $\gamma = 0.99$) muestra valores Q más elevados y un patrón de acciones más coherente, con flechas que reflejan claramente el intento de desplazarse hacia la posición del pasajero. Además, los valores Q del mejor agente son ligeramente más altos y uniformes que los obtenidos con SARSA en la misma configuración, lo que refleja el carácter optimista de Q-Learning al estimar siempre el valor de la acción óptima.

## Visualización del agente en acción

Al igual que analizamos las métricas numéricas, es fundamental observar cualitativamente cómo se comporta el agente tras el entrenamiento. A continuación, ejecutamos un episodio utilizando la tabla Q aprendida (explotación pura, sin exploración) y capturamos los fotogramas para generar un GIF.

Esta comprobación final valida que la política determinista aprendida ejecuta los transportes utilizando los caminos mínimos, consolidando el éxito del método.

In [ ]:
# Ejecutar un episodio utilizando la política greedy del mejor agente
best_agent = agents["α=0.5, γ=0.99"]
reward, len_episode, frames = capture_optimal_behavior(best_agent)

print(f"Episodio de prueba finalizado.")
print(f"Recompensa total: {reward}")
print(f"Pasos realizados: {len_episode}")

# Crear y mostrar el GIF
path_gif = "QL_Taxi_Optimal.gif"
create_gif_from_frames(frames, path_gif)
print("Archivo guardado en:", path_gif)
show_gif_in_notebook(path_gif)

### ¿Por qué la gráfica $f(t) = len(episodio_t)$ es un buen indicador de aprendizaje?

Hemos incluido la evolución temporal de la longitud del episodio (pasos por episodio). Esta métrica es un indicador de aprendizaje excelente, a veces incluso superior a la recompensa bruta, por dos motivos:

1. **Eficiencia de la ruta:** En entornos como *Taxi-v3*, donde cada movimiento penaliza con -1, la única forma de maximizar la recompensa es minimizando los pasos. La curva de longitud nos muestra visualmente cómo el agente deja de deambular de forma aleatoria (exploración) y comienza a trazar la ruta más corta hacia su objetivo.

2. **Aislamiento del ruido:** La recompensa total puede dar saltos bruscos (por ejemplo, el +20 al dejar al pasajero o el -10 por un error). Sin embargo, la métrica de pasos decrece de forma mucho más suave y monótona, permitiendo observar con mayor precisión en qué momento exacto convergen las políticas. En el caso de Q-Learning, la convergencia de la longitud de episodio suele ser ligeramente más suave que la de SARSA, al no inyectar el ruido exploratorio en las actualizaciones de valor.

## Posibles mejoras y líneas de investigación futuras

**Comparación en entornos estocásticos:** En un entorno determinista como Taxi-v3, las diferencias entre SARSA y Q-Learning se difuminan. Sería especialmente revelador comparar ambos algoritmos en entornos como *CliffWalking*, donde la naturaleza On-Policy de SARSA podría ofrecer ventajas en términos de seguridad (evitando los acantilados), mientras que Q-Learning podría converger más rápido pero con mayor riesgo durante el entrenamiento.

**Double Q-Learning:** Para combatir el sesgo optimista inherente a Q-Learning (sobreestimación sistemática de los valores Q debido al operador $\max$), se podría implementar y comparar la variante Double Q-Learning, que mantiene dos tablas Q independientes para desacoplar la selección y evaluación de acciones.

**Esquemas adaptativos de $\alpha$:** La tasa de aprendizaje constante funciona bien en Taxi-v3 por su naturaleza determinista, pero en entornos con ruido podría beneficiarse de esquemas adaptativos que reduzcan $\alpha$ conforme el agente gana confianza en sus estimaciones.

**Aumento del número de episodios y análisis estadístico:** Repetir cada configuración con distintas semillas aleatorias permitiría un análisis estadístico más riguroso, con media y desviación estándar del rendimiento, reduciendo el impacto de la estocasticidad.